In [25]:
import numpy as np
import fitz                   # was: import fitz  (same API, renamed)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import Normalizer

PDF = "nepal.pdf"
QUESTION = "what was the economic growth rate?"
N_RETRIEVE, N_KEEP = 100, 10

chunks = []
for pno , page in enumerate(fitz.open(PDF),1):
    w = page.get_text().split()
    for i in range (0 , len(w) , 150):
        chunks.append({"page": pno, "text": " ".join(w[i:i + 180])})
texts = [c["text"] for c in chunks]


lsa = make_pipeline(TfidfVectorizer(stop_words="english"),
                    TruncatedSVD(n_components=100),
                    Normalizer())
vecs = lsa.fit_transform(texts)
q = lsa.transform([QUESTION])[0]

cand = np.argsort(-(vecs @ q))[:N_RETRIEVE]



#Re rank : 
tfidf = TfidfVectorizer(stop_words="english", sublinear_tf=True)
C = tfidf.fit_transform([texts[i] for i in cand])      # fit on candidates only
qv = tfidf.transform([QUESTION])
rescore = (C @ qv.T).toarray().ravel()
final = cand[np.argsort(-rescore)[:N_KEEP]]

for rank, i in enumerate(final, 1):
    moved = list(cand).index(i) + 1
    print(f"{rank:2d}. (was #{moved:3d})  p.{chunks[i]['page']}  {chunks[i]['text'][:70]}")



 1. (was #  1)  p.1  Economic Survey 2023/24 Government of Nepal Ministry of Finance Singh 
 2. (was #  3)  p.371  671.4 Annual Growth Rate 8.46 5.97 14.44 6.19 9.30 9.40 1.60 6.65 9.90
 3. (was #  7)  p.367  Annual Growth Rate (Percentage) 6.03 6.27 2.61 1.73 6.20 6.90 7.61 9.5
 4. (was # 10)  p.357  Annual Growth Rate (%) 7.20 9.90 4.50 4.20 4.60 6.15 3.60 6.32 7.74 6.
 5. (was # 17)  p.314  5.17 2.61 5.16 2.43 2.85 2.35 2.76 3.05 Non-Agriculture 4.63 0.04 10.1
 6. (was # 13)  p.2  Economic Survey 2023/24 Government of Nepal Ministry of Finance Singh 
 7. (was # 32)  p.337  271.74 249.34 04.7 Other Industries 164.69 175.88 285.42 214.82 152.05
 8. (was #  2)  p.379  7.57 8.42 8.98 9.54 10.34 10.91 10.64 10.03 9.94 9.35 8.77 Source: Nep
 9. (was # 16)  p.304  Population (Million) 28.0 27.6 27.9 28.1 28.4 28.7 28.9 29.2 29.5 29.7
10. (was # 38)  p.423  rain around NPR 5 hundered thousand has loss in suder paschim province
